In [0]:
import pyspark.sql.functions as F
from pyspark.sql import DataFrame
from pyspark.sql.types import DoubleType, FloatType

silver = spark.table("workspace.silver.gho_clean")
required = ["country_code", "region", "year", "indicator_code", "numeric_value"]

In [0]:
def _is_missing(df: DataFrame, column: str):
    col = F.col(column)
    if isinstance(df.schema[column].dataType, (DoubleType, FloatType)):
        return col.isNull() | F.isnan(col)
    return col.isNull()


def count_missing(df: DataFrame, columns: list[str]) -> dict[str, int]:
    row = df.agg(
        *[F.sum(_is_missing(df, c).cast("int")).alias(c) for c in columns]
    ).first()
    return {c: row[c] or 0 for c in columns}


def assert_no_nulls(df: DataFrame, columns: list[str], table_name: str) -> None:
    failures = {c: n for c, n in count_missing(df, columns).items() if n > 0}
    assert not failures, f"{table_name}: nulls in required columns: {failures}"


def assert_not_all_null(df: DataFrame, columns: list[str], table_name: str) -> None:
    total = df.count()
    empty = [c for c, n in count_missing(df, columns).items() if n == total]
    assert not empty, f"{table_name}: columns entirely null: {empty}"


def report_null_rates(df: DataFrame, columns: list[str]) -> DataFrame:
    total = df.count()
    rows = [(c, n, n / total if total else 0.0) for c, n in count_missing(df, columns).items()]
    return spark.createDataFrame(rows, "column string, null_count long, null_rate double")

In [0]:
n_duplicates = (silver.groupBy("country_code", "year", "indicator_code")
    .count().filter("count > 1").count())
assert n_duplicates == 0, f"{n_duplicates} duplicated keys"

In [0]:
n_indicators = silver.select("indicator_code").distinct().count()
assert n_indicators == 15, f"{n_indicators} indicators, expected 15"

In [0]:
dtype = dict(silver.dtypes)["numeric_value"]
assert dtype == "double", f"numeric_value is {dtype}, expected double"

In [0]:
row = silver.select([
    F.avg((F.col(c).isNull() | (F.trim(F.col(c).cast("string")) == "")).cast("int")).alias(c)
    for c in required
]).first()
assert row is not None, "silver is empty"
failures = {c: round(rate, 4) for c, rate in row.asDict().items() if rate > 0}
assert not failures, f"Nulls/empty strings in required columns: {failures}"

In [0]:
silver.printSchema()